TODO: 
- compare results against CandidateEdgeSampler
- prestoring the candidates
- see how many posts on average between a 24 hour window

In [1]:
# Cell 1: Imports and Setup
import logging
import time
import sys
import os
import numpy as np
import warnings
import json
import torch
import torch.nn as nn
import pickle
import pandas as pd
from datetime import datetime, timedelta
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()

# Suppress warnings and matplotlib debug messages
warnings.filterwarnings('ignore')
logging.getLogger('matplotlib').setLevel(logging.WARNING)

# Append parent directory to path if running in notebook
import sys
sys.path.append("..")  # Add parent directory to path for imports

# These imports will work once parent directory is in path
from models.TGAT import TGAT
from models.GraphRec import GraphRec
from models.GraphRecMulti import GraphRecMulti
from models.GraphRecMultiCo import GraphRecMultiCo
from models.modules import MergeLayer
from utils.utils import set_random_seed, convert_to_gpu, get_parameter_sizes
from utils.utils import get_neighbor_sampler, CandidateEdgeSampler
from utils.DataLoader import get_idx_data_loader, get_link_prediction_data, get_link_prediction_data_eval
from utils.EarlyStopping import EarlyStopping
from utils.load_configs import get_link_prediction_args

# Make sure the directory exists for saving results
os.makedirs("./notebook_results", exist_ok=True)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/home/a5wu/.conda/envs/bluesky/lib/python3.11/runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/home/a5wu/.conda/envs/bluesky/lib/python3.11/runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "/home/a5wu/.conda/envs/bluesky/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/a5wu/.conda/envs/bluesky/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_in

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/home/a5wu/.conda/envs/bluesky/lib/python3.11/runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/home/a5wu/.conda/envs/bluesky/lib/python3.11/runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "/home/a5wu/.conda/envs/bluesky/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/a5wu/.conda/envs/bluesky/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_in

AttributeError: _ARRAY_API not found

In [2]:
# Cell 2: Create Mock Arguments
class Args:
    def __init__(self):
        # Dataset configuration
        self.dataset_name = "bluesky"
        self.val_ratio = 0.15
        self.test_ratio = 0.15
        
        # Model configuration
        self.model_name = "GraphRecMultiCo"
        self.time_feat_dim = 100
        self.channel_embedding_dim = 50  # Default from load_configs.py
        self.patch_size = 5  # From command line
        self.num_layers = 2
        self.num_heads = 2  # From command line
        self.dropout = 0.1
        self.max_input_sequence_length = 32
        
        # Training configuration
        self.batch_size = 4  # From command line
        self.num_neighbors = 10  # From command line
        self.time_gap = 2000
        self.walk_length = 2  # From command line
        
        # Sampling configuration
        self.sample_neighbor_strategy = "recent"  # Default from load_configs.py
        self.time_scaling_factor = 1e-6  # Default from load_configs.py
        
        # Evaluation configuration
        self.negative_sample_strategy = "real"  # From command line
        self.gpu = 0  # From command line
        self.device = torch.device(f'cuda:{self.gpu}' if torch.cuda.is_available() and self.gpu >= 0 else 'cpu')
        self.seed = 100  # From command line
        self.num_runs = 1  # From command line
        
        # Model loading configuration
        self.load_model_name = f'{self.model_name}_seed{self.seed}_3'
        self.save_result_name = f'{self.negative_sample_strategy}_negative_sampling_{self.model_name}_seed{self.seed}'

args = Args()
print(f"Using device: {args.device}")
print(f"Model: {args.model_name}")
print(f"Negative sample strategy: {args.negative_sample_strategy}")

Using device: cuda:0
Model: GraphRecMultiCo
Negative sample strategy: real


In [3]:
# Cell 3: Load Data
print("Loading data...")
# Get data for training, validation and testing
node_raw_features, _, full_data, _, eval_test_data, dynamic_user_features, post_dynamic_features = \
    get_link_prediction_data_eval(dataset_name=args.dataset_name, val_ratio=args.val_ratio, test_ratio=args.test_ratio)

# Initialize validation and test neighbor sampler to retrieve temporal graph
full_neighbor_sampler = get_neighbor_sampler(data=full_data, 
                                             sample_neighbor_strategy="recent",  # You can change this as needed
                                             time_scaling_factor=1.0, 
                                             seed=1)

# Create data loader for testing
test_idx_data_loader = get_idx_data_loader(
    indices_list=list(range(len(eval_test_data.src_node_ids))), 
    batch_size=args.batch_size, 
    shuffle=False
)

print(f"Loaded data with {len(full_data.src_node_ids)} interactions")
print(f"Test data has {len(eval_test_data.src_node_ids)} interactions")

Loading data...
val_time: 2023-06-14 17:45:07
test_time: 2023-06-23 19:47:26
The dataset has 22131398 interactions, involving 5972593 different nodes
The new node test dataset has 4360 interactions, involving 7325 different nodes
597259 nodes were used for the inductive testing, i.e. are never seen during training
Loaded data with 22131398 interactions
Test data has 4360 interactions


In [4]:
_ = None
_

In [5]:
# Cell 4: Load Post Embeddings
print("Loading post embeddings...")

# Try to load from parquet first (faster)
post_embeddings_path = os.path.join(os.path.expanduser("~"), 'post_dynamic_embeddings.parquet')
post_embeddings_df = pd.read_parquet(post_embeddings_path)
print(f"Loaded {len(post_embeddings_df)} post embeddings from parquet file")

# Display sample of post embeddings
print("Sample of post embeddings DataFrame:")
print(post_embeddings_df.head())

# Check for any issues in the data
print("\nDataFrame info:")
print(post_embeddings_df.info())

# Verify embedding dimensions
sample_embedding = post_embeddings_df['embedding'].iloc[0]
print(f"\nSample embedding dimension: {sample_embedding.shape}")

Loading post embeddings...
Loaded 26266969 post embeddings from parquet file
Sample of post embeddings DataFrame:
          post_id  user_id               timestamp  \
11481610  1083355    28292 2023-03-15 00:00:00.000   
11481603  1083354    28292 2023-03-15 00:00:00.000   
17656846  2374256    28292 2023-03-15 00:00:08.000   
25319257  5450300    32529 2023-03-15 00:00:13.305   
22883522  4254353    59091 2023-03-15 00:00:41.000   

                                                  embedding  num_interactions  \
11481610  [-0.1958, 0.1721, 0.1202, 0.5557, -0.177, -0.4...                 1   
11481603  [-0.1958, 0.1721, 0.1202, 0.5557, -0.177, -0.4...                 1   
17656846  [-0.1958, 0.1721, 0.1202, 0.5557, -0.177, -0.4...                 1   
25319257  [-0.1786, 0.218, -0.1038, -0.4153, -0.03143, -...                 0   
22883522  [-0.2205, 0.304, 0.0451, -0.5635, -0.0312, -0....                 1   

         embedding_source                                     prev_embeddi

In [6]:
# Cell 5: Implement EmbeddingCandidateEdgeSampler
class EmbeddingCandidateEdgeSampler:
    """
    Candidate edge sampler that uses embedding similarity for candidate generation.
    """
    def __init__(self, user_dynamic_features, post_embeddings_df, time_window_hours=24, 
                 n_candidates=100, seed=None, include_true_dst=True):
        """
        Initialize the embedding-based candidate sampler.
        
        Args:
            user_dynamic_features: Dictionary of user embeddings
            post_embeddings_df: DataFrame with post embeddings
            time_window_hours: Hours to look back for post candidates
            n_candidates: Number of candidates to return
            seed: Random seed for reproducibility
            include_true_dst: Whether to include the true destination in candidates
        """
        self.logger = logging.getLogger(__name__)
        
        # Store the user dynamic features directly without adjustment
        self.user_dynamic_features = user_dynamic_features
        
        self.post_embeddings_df = post_embeddings_df
        self.time_window_hours = time_window_hours
        self.n_candidates = n_candidates
        self.seed = seed
        self.include_true_dst = include_true_dst
        
        # # Convert user_dynamic_features to DataFrame for easier access
        # self.user_dynamic_features_df = pd.DataFrame.from_dict(self.user_dynamic_features, orient='index')
        # self.user_dynamic_features_df.index = pd.to_datetime(self.user_dynamic_features_df.index, unit='s')
        # self.user_dynamic_features_df = self.user_dynamic_features_df.sort_index()
            
        self.logger.info(f"Initialized EmbeddingCandidateEdgeSampler with {len(self.post_embeddings_df)} post embeddings")
        
        # Set random seed if provided
        self.reset_random_state()
        
        # Cache for post embeddings by day to speed up retrieval
        self.post_embeddings_cache = {}
        
        # Debug counters
        self.true_post_added_count = 0
        self.total_processed = 0
        
        # Detailed fallback counters
        self.fallback_counters = {
            "user_embedding_not_available": 0,
            "embedding_date_not_found": 0,
            "user_id_not_found": 0,
            "no_active_posts": 0,
            "exception_occurred": 0
        }
        
        # Hit rate tracking
        self.hit_counters = {20: 0, 50: 0, 100: 0, 500: 0, 1000: 0, 2000: 0, 3000: 0, 5000: 0}
        self.k_values = sorted(self.hit_counters.keys())
    
    def reset_random_state(self):
        """Reset random state for reproducibility during evaluation"""
        if self.seed is not None:
            np.random.seed(self.seed)
    
    def sample(self, size, batch_src_node_ids, batch_dst_node_ids, batch_node_interact_times, 
               current_batch_start_time=None, popularity_based=False):
        """
        Sample candidate edges for each interaction.
        
        Args:
            size: Number of interactions to sample for
            batch_src_node_ids: Source node IDs (users)
            batch_dst_node_ids: Destination node IDs (posts that users interacted with)
            batch_node_interact_times: Timestamps of interactions
            current_batch_start_time: Not used, kept for compatibility
            popularity_based: Whether to use popularity-based sampling (fallback)
        
        Returns:
            Dictionary mapping interaction times to candidate post IDs
        """
        candidates_dict = {}
        debug_info = []  # For debugging
        
        # Process each interaction
        for i in range(size):
            self.total_processed += 1
            user_id = batch_src_node_ids[i]
            timestamp = pd.Timestamp(batch_node_interact_times[i], unit='s')
            true_post_id = batch_dst_node_ids[i]
            
            # Get embedding date (7am of the day)
            embedding_date = pd.Timestamp(timestamp.date()) + pd.Timedelta(hours=7)
            embedding_date_int = int(embedding_date.timestamp())
            
            # For debugging
            user_info = {
                "user_id": user_id,
                "timestamp": timestamp,
                "true_post_id": true_post_id,
                "embedding_date": embedding_date
            }
            
            # Get user embedding
            try:
                # Check if embedding date exists
                if embedding_date_int not in self.user_dynamic_features:
                    user_info["error"] = "Embedding date not found"
                    debug_info.append(user_info)
                    self.fallback_counters["embedding_date_not_found"] += 1
                    random_candidates = np.random.choice(
                        self.post_embeddings_df['post_id'].unique(), 
                        size=self.n_candidates, 
                        replace=False
                    )
                    candidates_dict[batch_node_interact_times[i]] = random_candidates
                    continue
                
                # Check if user ID exists in the date's dictionary
                if user_id not in self.user_dynamic_features[embedding_date_int]:
                    user_info["error"] = "User ID not found"
                    debug_info.append(user_info)
                    self.fallback_counters["user_id_not_found"] += 1
                    random_candidates = np.random.choice(
                        self.post_embeddings_df['post_id'].unique(), 
                        size=self.n_candidates, 
                        replace=False
                    )
                    candidates_dict[batch_node_interact_times[i]] = random_candidates
                    continue
                
                # Get user embedding directly from the nested dictionary
                user_embedding = self.user_dynamic_features[embedding_date_int][user_id]
                
                # # Print user embedding info
                # print("user_embedding type: ", type(user_embedding))
                # print("user_embedding: ", user_embedding)
                # print("user_info: ", user_info)

                
                # Skip if user embedding is not available
                if not isinstance(user_embedding, np.ndarray):
                    user_info["error"] = "User embedding not available"
                    debug_info.append(user_info)
                    self.fallback_counters["user_embedding_not_available"] += 1
                    random_candidates = np.random.choice(
                        self.post_embeddings_df['post_id'].unique(), 
                        size=self.n_candidates, 
                        replace=False
                    )
                    candidates_dict[batch_node_interact_times[i]] = random_candidates
                    continue
                    
                # Get posts active within time window
                time_window_start = timestamp - timedelta(hours=self.time_window_hours)
                
                # Use cache for post embeddings if available
                day_key = timestamp.date().isoformat()
                if day_key in self.post_embeddings_cache:
                    active_posts = self.post_embeddings_cache[day_key]
                else:
                    active_posts = self.post_embeddings_df[
                        (self.post_embeddings_df['timestamp'] < timestamp) & 
                        (self.post_embeddings_df['timestamp'] >= time_window_start)
                    ]
                    self.post_embeddings_cache[day_key] = active_posts
                
                user_info["num_active_posts"] = len(active_posts)
                
                if len(active_posts) == 0:
                    user_info["error"] = "No active posts in time window"
                    debug_info.append(user_info)
                    self.fallback_counters["no_active_posts"] += 1
                    random_candidates = np.random.choice(
                        self.post_embeddings_df['post_id'].unique(), 
                        size=self.n_candidates, 
                        replace=False
                    )
                    candidates_dict[batch_node_interact_times[i]] = random_candidates
                    continue
                
                # Get latest embedding for each post
                latest_embeddings = (
                    active_posts.sort_values('timestamp')
                    .groupby('post_id')
                    .last()
                    .reset_index()
                )
                
                # Calculate similarities
                post_embeddings = np.stack(latest_embeddings['embedding'].values)
                similarities = cosine_similarity([user_embedding], post_embeddings)[0]
                
                # Get top N candidates
                top_indices = np.argsort(similarities)[-self.n_candidates:][::-1]
                candidate_posts = latest_embeddings.iloc[top_indices]['post_id'].values
                top_similarities = similarities[top_indices]
                
                # Check if true post is in candidates and track it
                true_post_in_candidates = true_post_id in candidate_posts
                user_info["true_post_in_candidates"] = true_post_in_candidates
                
                # Find position of true post in the ranked list (if present)
                true_post_position = None
                for idx, post_id in enumerate(candidate_posts):
                    if post_id == true_post_id:
                        true_post_position = idx
                        break
                
                # Update hit counters for each k value
                if true_post_position is not None:
                    for k in self.k_values:
                        if true_post_position < k:
                            self.hit_counters[k] += 1
                
                # Make sure true post is in candidates for evaluation if needed
                if self.include_true_dst and not true_post_in_candidates:
                    # Replace the last candidate with the true post
                    candidate_posts[-1] = true_post_id
                    self.true_post_added_count += 1
                    user_info["true_post_added"] = True
                
                user_info["top_similarity"] = float(top_similarities[0]) if len(top_similarities) > 0 else None
                debug_info.append(user_info)
                    
                candidates_dict[batch_node_interact_times[i]] = candidate_posts
                
            except Exception as e:
                user_info["error"] = f"Exception: {str(e)}"
                debug_info.append(user_info)
                self.fallback_counters["exception_occurred"] += 1
                random_candidates = np.random.choice(
                    self.post_embeddings_df['post_id'].unique(), 
                    size=self.n_candidates, 
                    replace=False
                )
                candidates_dict[batch_node_interact_times[i]] = random_candidates
        
        # Save debug info for analysis
        self.debug_info = debug_info
        
        # Print debug statistics
        if self.total_processed % 100 == 0:
            print(f"Debug stats: Total processed: {self.total_processed}")
            print(f"True post added count: {self.true_post_added_count} ({self.true_post_added_count/self.total_processed*100:.2f}%)")
            
            # Print hit rate statistics
            print("Hit Rate@k:")
            for k in self.k_values:
                hit_rate = (self.hit_counters[k] / self.total_processed) * 100
                print(f"  Hit@{k}: {hit_rate:.2f}%")
            
            # Print detailed fallback statistics
            total_fallbacks = sum(self.fallback_counters.values())
            print(f"Total fallbacks: {total_fallbacks} ({total_fallbacks/self.total_processed*100:.2f}%)")
            print("Fallback reasons breakdown:")
            for reason, count in self.fallback_counters.items():
                if count > 0:
                    print(f"  - {reason}: {count} ({count/total_fallbacks*100:.2f}% of fallbacks)")
            
            # Analyze why true posts aren't in candidates
            if len(debug_info) > 0:
                not_in_candidates = [info for info in debug_info if info.get("true_post_in_candidates") is False]
                if not_in_candidates:
                    print(f"Sample reasons true post not in candidates:")
                    for i, info in enumerate(not_in_candidates[:3]):
                        print(f"  Example {i+1}: {info.get('error', 'No error')}, Active posts: {info.get('num_active_posts', 'N/A')}")
        
        return candidates_dict

# Create the embedding-based candidate sampler
embedding_sampler = EmbeddingCandidateEdgeSampler(
    user_dynamic_features=dynamic_user_features,
    post_embeddings_df=post_embeddings_df,
    time_window_hours=24,  # Consider increasing this to capture more posts
    n_candidates=3000,
    seed=args.seed
)

print(f"Created embedding-based candidate sampler with {len(post_embeddings_df)} post embeddings")

INFO:__main__:Initialized EmbeddingCandidateEdgeSampler with 26266969 post embeddings


Created embedding-based candidate sampler with 26266969 post embeddings


In [7]:
# Cell 6: Load Pre-trained Model
print(f"Loading pre-trained {args.model_name} model...")

# Set random seed for reproducibility
set_random_seed(seed=args.seed)

# Create model
if args.model_name == 'GraphRec':
    dynamic_backbone = GraphRec(node_raw_features=node_raw_features, 
                                neighbor_sampler=full_neighbor_sampler,
                                time_feat_dim=args.time_feat_dim, 
                                channel_embedding_dim=args.channel_embedding_dim, 
                                patch_size=args.patch_size,
                                num_layers=args.num_layers, 
                                num_heads=args.num_heads, 
                                dropout=args.dropout,
                                max_input_sequence_length=args.max_input_sequence_length, 
                                device=args.device, 
                                user_dynamic_features=dynamic_user_features, 
                                src_max_id=eval_test_data.src_max_id)
elif args.model_name == 'GraphRecMulti':
    dynamic_backbone = GraphRecMulti(node_raw_features=node_raw_features, 
                                    neighbor_sampler=full_neighbor_sampler,
                                    time_feat_dim=args.time_feat_dim, 
                                    channel_embedding_dim=args.channel_embedding_dim, 
                                    patch_size=args.patch_size,
                                    num_layers=args.num_layers, 
                                    num_heads=args.num_heads, 
                                    dropout=args.dropout,
                                    max_input_sequence_length=args.max_input_sequence_length, 
                                    device=args.device, 
                                    user_dynamic_features=dynamic_user_features, 
                                    src_max_id=eval_test_data.src_max_id)
elif args.model_name == 'GraphRecMultiCo':
    dynamic_backbone = GraphRecMultiCo(node_raw_features=node_raw_features, 
                                    neighbor_sampler=full_neighbor_sampler,
                                    time_feat_dim=args.time_feat_dim, 
                                    channel_embedding_dim=args.channel_embedding_dim, 
                                    patch_size=args.patch_size,
                                    num_layers=args.num_layers, 
                                    num_heads=args.num_heads, 
                                    dropout=args.dropout,
                                    max_input_sequence_length=args.max_input_sequence_length, 
                                    device=args.device, 
                                    user_dynamic_features=dynamic_user_features,
                                    post_dynamic_features=post_dynamic_features,
                                    src_max_id=eval_test_data.src_max_id, 
                                    walk_length=args.walk_length, 
                                    num_neighbors=args.num_neighbors)
elif args.model_name == 'TGAT':
    dynamic_backbone = TGAT(node_raw_features=node_raw_features, 
                            edge_raw_features=edge_raw_features, 
                            neighbor_sampler=full_neighbor_sampler,
                            time_feat_dim=args.time_feat_dim, 
                            num_layers=args.num_layers, 
                            dropout=args.dropout, 
                            device=args.device)
else:
    raise ValueError(f"Wrong value for model_name {args.model_name}!")

link_predictor = MergeLayer(input_dim1=node_raw_features.shape[1]+64, 
                            input_dim2=node_raw_features.shape[1]+64,
                            hidden_dim=node_raw_features.shape[1]+64, 
                            output_dim=1)
model = nn.Sequential(dynamic_backbone, link_predictor)

print(f'Model: {args.model_name}, #parameters: {get_parameter_sizes(model) * 4 / 1024 / 1024:.2f} MB')

# Try to load the saved model
try:
    load_model_folder = f"./saved_models/{args.model_name}/{args.dataset_name}/{args.load_model_name}"
    early_stopping = EarlyStopping(patience=0, 
                                  save_model_folder=load_model_folder,
                                  save_model_name=args.load_model_name, 
                                  logger=logger, 
                                  model_name=args.model_name)
    early_stopping.load_checkpoint(model, map_location='cpu')
    print(f"Successfully loaded model from {load_model_folder}")
except Exception as e:
    print(f"Warning: Could not load pre-trained model: {str(e)}")
    print("Continuing with untrained model for testing purposes...")

# Move model to device
model = convert_to_gpu(model, device=args.device)

Loading pre-trained GraphRecMultiCo model...


INFO:root:load model ./saved_models/GraphRecMultiCo/bluesky/GraphRecMultiCo_seed100_3/GraphRecMultiCo_seed100_3.pkl


Model: GraphRecMultiCo, #parameters: 2.81 MB
Successfully loaded model from ./saved_models/GraphRecMultiCo/bluesky/GraphRecMultiCo_seed100_3


In [10]:
# Cell 7: Modified Evaluation Function
def evaluate_with_embedding_candidates(model_name, model, neighbor_sampler, evaluate_idx_data_loader,
                                      evaluate_neg_edge_sampler, evaluate_data,
                                      num_neighbors=20, time_gap=8, max_samples=None):
    """
    Evaluate models using embedding-based candidate generation
    
    Args:
        model_name: Name of the model
        model: Model to evaluate
        neighbor_sampler: Neighbor sampler
        evaluate_idx_data_loader: Data loader for evaluation indices
        evaluate_neg_edge_sampler: Candidate edge sampler (our embedding-based sampler)
        evaluate_data: Evaluation data
        num_neighbors: Number of neighbors to sample
        time_gap: Time gap for neighbor sampling
        max_samples: Maximum number of samples to evaluate (for debugging)
    
    Returns:
        Average MRR score
    """
    model[0].set_neighbor_sampler(neighbor_sampler)
    model.eval()
    candidates_length = {}
    recommended_posts = []

    with torch.no_grad():
        # Store evaluation metrics
        mrr_results = []
        sample_count = 0
        
        evaluate_idx_data_loader_tqdm = tqdm(evaluate_idx_data_loader, ncols=120)
        for batch_idx, evaluate_data_indices in enumerate(evaluate_idx_data_loader_tqdm):
            # Early stopping for debugging
            if max_samples is not None and sample_count >= max_samples:
                break
                
            evaluate_data_indices = evaluate_data_indices.numpy()
            batch_src_node_ids, batch_dst_node_ids, batch_node_interact_times, batch_edge_ids = \
                evaluate_data.src_node_ids[evaluate_data_indices], evaluate_data.dst_node_ids[evaluate_data_indices], \
                evaluate_data.node_interact_times[evaluate_data_indices], evaluate_data.edge_ids[evaluate_data_indices]
            
            # For dynamic features
            batch_src_idx = evaluate_data.idx[evaluate_data_indices]
            
            # Get candidates using embedding-based sampler
            candidates_dict = evaluate_neg_edge_sampler.sample(
                len(batch_src_node_ids), 
                batch_src_node_ids, 
                batch_dst_node_ids, 
                batch_node_interact_times
            )
            
            sample_count += len(batch_src_node_ids)

            # Iterate through candidates_dict to calculate lengths
            for start_time, candidates in candidates_dict.items():
                # Store in candidates_length
                start_time = str(start_time)
                if start_time not in candidates_length:
                    num_candidates = len(candidates)
                    candidates_length[start_time] = num_candidates

            # Prepare for batch processing
            batch_candidates = []
            batch_interact_times = []
            batch_src_ids = []
            batch_src_ids_no_duplicates = []
            batch_idx = []

            for src_id, interact_time, src_idx, true_dst_id in zip(
                batch_src_node_ids, batch_node_interact_times, batch_src_idx, batch_dst_node_ids
            ):
                candidate_ids = candidates_dict[interact_time]
                batch_candidates.append(list(candidate_ids))
                batch_interact_times.append([interact_time] * len(candidate_ids))
                batch_src_ids.append([src_id] * len(candidate_ids))
                batch_src_ids_no_duplicates.append(src_id)
                batch_idx.append([src_idx] * len(candidate_ids))

            # Flatten batch data for processing
            batch_candidates = np.concatenate(batch_candidates)
            batch_interact_times = np.concatenate(batch_interact_times)
            batch_src_ids = np.concatenate(batch_src_ids)
            batch_idx = np.concatenate(batch_idx)

            if model_name in {'GraphRec', 'GraphRecMulti', 'GraphRecMultiCo'}:
                # Compute embeddings in one operation
                src_embeddings, dst_embeddings = model[0].compute_src_dst_node_temporal_embeddings(
                    src_node_ids=batch_src_ids,
                    dst_node_ids=batch_candidates,
                    node_interact_times=batch_interact_times,
                    batch_idx=batch_idx
                )
            elif model_name == 'TGAT':
                # Compute embeddings in one operation
                src_embeddings, dst_embeddings = model[0].compute_src_dst_node_temporal_embeddings(
                    src_node_ids=batch_src_ids,
                    dst_node_ids=batch_candidates,
                    node_interact_times=batch_interact_times,
                    num_neighbors=num_neighbors
                )
            else:
                raise ValueError(f"Wrong value for model_name {model_name}!")

            # Compute scores for all user-candidate pairs in the batch
            probabilities = model[1](input_1=src_embeddings, input_2=dst_embeddings).squeeze(dim=-1).sigmoid()

            # Reshape probabilities to group by users
            split_indices = np.cumsum([len(candidates_dict[interact_time]) for interact_time in batch_node_interact_times])
            grouped_probabilities = np.split(probabilities.cpu().numpy(), split_indices)
            grouped_candidates = np.split(batch_candidates, split_indices)

            # Evaluate MRR for each user in the batch
            for post_probabilities, post_candidates, true_dst_id, src_id in zip(
                grouped_probabilities, grouped_candidates, batch_dst_node_ids, batch_src_ids_no_duplicates
            ):
                # Convert to numpy for indexing
                post_probabilities = np.array(post_probabilities)
                post_candidates = np.array(post_candidates)
                
                # Find the index of the true destination ID
                true_dst_index = np.where(post_candidates == true_dst_id)[0]
                
                if len(true_dst_index) > 0:  # Ensure the true destination exists
                    true_dst_index = true_dst_index[0]
                    true_dst_probability = post_probabilities[true_dst_index]
                    
                    # Count how many probabilities are higher than the true_dst_probability
                    rank = 1 + np.sum(post_probabilities > true_dst_probability)
                    mrr_results.append(1 / rank)
                else:
                    # True destination not found in candidates
                    mrr_results.append(0)

                # Sort candidates by probability for recommendation list
                sorted_indices = np.argsort(-post_probabilities) 
                sorted_candidates = post_candidates[sorted_indices]
                recommended_posts.append(sorted_candidates.tolist())
                
            # Update progress bar
            evaluate_idx_data_loader_tqdm.set_description(
                f'Batch {batch_idx+1}, MRR so far: {np.mean(mrr_results):.4f}'
            )

    # Save results
    os.makedirs(f"./notebook_results/{model_name}/bluesky", exist_ok=True)
    
    # Save recommended posts
    with open(f"./notebook_results/{model_name}/bluesky/recommended_posts.json", "w") as json_file:
        json.dump(recommended_posts, json_file, indent=4)

    # Save MRR results
    np.save(f"./notebook_results/{model_name}/bluesky/mrr_results.npy", np.array(mrr_results))
    
    # Calculate average MRR
    avg_mrr = np.mean(mrr_results)
    print(f"Mean Reciprocal Rank (MRR): {avg_mrr:.4f}")
    
    # Save candidate lengths
    with open(f"./notebook_results/{model_name}/bluesky/candidates_length.json", 'w') as f:
        json.dump(candidates_length, f, indent=4)
    
    # Return debug info from sampler along with MRR
    return avg_mrr, mrr_results, embedding_sampler.debug_info

In [11]:
# Cell 8: Run Full Evaluation on the Entire Dataset
print("Running full evaluation on all samples...")

# Remove the max_samples parameter or set it to None to process all samples
test_mrr, test_mrr_results, debug_info = evaluate_with_embedding_candidates(
    model_name=args.model_name,
    model=model,
    neighbor_sampler=full_neighbor_sampler,
    evaluate_idx_data_loader=test_idx_data_loader,
    evaluate_neg_edge_sampler=embedding_sampler,
    evaluate_data=eval_test_data,
    num_neighbors=args.num_neighbors,
    time_gap=args.time_gap,
    max_samples=None  # Set to None to process all samples
)

print(f"Test MRR (full evaluation): {test_mrr:.4f}")

# Save the results to files for later analysis
os.makedirs(f"./results/{args.model_name}", exist_ok=True)
np.save(f"./results/{args.model_name}/mrr_results_embedding_candidates.npy", np.array(test_mrr_results))

# Analyze debug information
debug_df = pd.DataFrame(debug_info)
print("\nDebug information summary:")
print(f"Number of samples: {len(debug_df)}")

# Save debug information
debug_df.to_csv(f"./results/{args.model_name}/embedding_candidates_debug.csv", index=False)

# Check if 'error' column exists before accessing it
if 'error' in debug_df.columns:
    print(f"Samples with errors: {debug_df['error'].notna().sum()} ({debug_df['error'].notna().sum()/len(debug_df)*100:.2f}%)")
else:
    print("No errors found in debug information")

if 'true_post_in_candidates' in debug_df.columns:
    print(f"True post in candidates: {debug_df['true_post_in_candidates'].sum()} out of {len(debug_df)} ({debug_df['true_post_in_candidates'].sum()/len(debug_df)*100:.2f}%)")
if 'top_similarity' in debug_df.columns:
    print(f"Average top similarity: {debug_df['top_similarity'].mean():.4f}")
if 'num_active_posts' in debug_df.columns:
    print(f"Average number of active posts: {debug_df['num_active_posts'].mean():.1f}")

# Display errors if any
if 'error' in debug_df.columns and debug_df['error'].notna().sum() > 0:
    print("\nTop 5 error types:")
    print(debug_df['error'].value_counts().head())

# Additional detailed analysis
print("\nHit Rate Analysis:")
for k in embedding_sampler.k_values:
    hit_rate = embedding_sampler.hit_counters[k] / embedding_sampler.total_processed * 100
    print(f"Hit@{k}: {hit_rate:.2f}%")

# Summary of fallbacks
fallbacks_total = sum(embedding_sampler.fallback_counters.values())
if fallbacks_total > 0:
    print(f"\nFallback Summary ({fallbacks_total} total, {fallbacks_total/embedding_sampler.total_processed*100:.2f}%):")
    for reason, count in embedding_sampler.fallback_counters.items():
        if count > 0:
            print(f"- {reason}: {count} ({count/fallbacks_total*100:.2f}%)")

Running full evaluation on all samples...


Batch [15522642 15522642 15522642 ... 15518071 15518071 15518071], MRR so far: 0.1920:   2%| | 23/1090 [01:11<55:01,  3.

Debug stats: Total processed: 100
True post added count: 30 (30.00%)
Hit Rate@k:
  Hit@20: 26.00%
  Hit@50: 31.00%
  Hit@100: 38.00%
  Hit@500: 50.00%
  Hit@1000: 57.00%
  Hit@2000: 65.00%
  Hit@3000: 70.00%
  Hit@5000: 70.00%
Total fallbacks: 0 (0.00%)
Fallback reasons breakdown:


Batch [15515803 15515803 15515803 ... 15508928 15508928 15508928], MRR so far: 0.1775:   4%| | 48/1090 [02:29<53:56,  3.

Debug stats: Total processed: 200
True post added count: 50 (25.00%)
Hit Rate@k:
  Hit@20: 25.00%
  Hit@50: 33.00%
  Hit@100: 40.50%
  Hit@500: 55.00%
  Hit@1000: 61.50%
  Hit@2000: 70.00%
  Hit@3000: 75.00%
  Hit@5000: 75.00%
Total fallbacks: 0 (0.00%)
Fallback reasons breakdown:
Sample reasons true post not in candidates:
  Example 1: No error, Active posts: 346444


...

Debug stats: Total processed: 4200
True post added count: 976 (23.24%)
Hit Rate@k:
  Hit@20: 23.86%
  Hit@50: 30.50%
  Hit@100: 35.95%
  Hit@500: 52.60%
  Hit@1000: 60.81%
  Hit@2000: 71.36%
  Hit@3000: 76.76%
  Hit@5000: 76.76%
Total fallbacks: 0 (0.00%)
Fallback reasons breakdown:
Sample reasons true post not in candidates:
  Example 1: No error, Active posts: 346444


Batch [15515430 15515430 15515430 ... 15511491 15511491 15511491], MRR so far: 0.1420:  98%|▉| 1073/1090 [54:54<00:51,  

Debug stats: Total processed: 4300
True post added count: 999 (23.23%)
Hit Rate@k:
  Hit@20: 23.86%
  Hit@50: 30.51%
  Hit@100: 35.95%
  Hit@500: 52.74%
  Hit@1000: 60.95%
  Hit@2000: 71.40%
  Hit@3000: 76.77%
  Hit@5000: 76.77%
Total fallbacks: 0 (0.00%)
Fallback reasons breakdown:


Batch [15523591 15523591 15523591 ... 15524777 15524777 15524777], MRR so far: 0.1417: 100%|█| 1090/1090 [55:45<00:00,  


Mean Reciprocal Rank (MRR): 0.1417
Test MRR (full evaluation): 0.1417

Debug information summary:
Number of samples: 4
No errors found in debug information
True post in candidates: 3 out of 4 (75.00%)
Average top similarity: 0.8701
Average number of active posts: 346444.0

Hit Rate Analysis:
Hit@20: 23.88%
Hit@50: 30.50%
Hit@100: 35.91%
Hit@500: 52.77%
Hit@1000: 61.00%
Hit@2000: 71.36%
Hit@3000: 76.74%
Hit@5000: 76.74%


In [ ]:
batch_src_node_ids

In [ ]:
embedding_sampler.true_post_added_count, embedding_sampler.total_processed, embedding_sampler.fallback_counters

In [ ]:
embedding_sampler.hit_counters

In [ ]:
# Create a DataFrame to display source nodes, destination nodes, and timestamps
sample_data = pd.DataFrame({
    'src_node_id': eval_test_data.src_node_ids[:10],
    'dst_node_id': eval_test_data.dst_node_ids[:10],
    'timestamp': [datetime.fromtimestamp(t) for t in eval_test_data.node_interact_times[:10]]
})
sample_data['adjusted_dst_id'] = sample_data['dst_node_id'] - 1
sample_data

In [ ]:
# let's get all the embeddings for post_id 111075 between 06-14 10:52 and 10:57
post_id = 111076
start_time = pd.Timestamp('2023-06-14 10:52:00')
end_time = pd.Timestamp('2023-06-14 10:57:00')

# Filter the post_embeddings_df for the given post_id and time range
filtered_data = post_embeddings_df[
    (post_embeddings_df['post_id'] == post_id) &
    (post_embeddings_df['timestamp'] >= start_time) &
    (post_embeddings_df['timestamp'] <= end_time)
]

# Display the filtered data
filtered_data

In [ ]:
# Join post_embeddings_df with sample_data
# Convert post_id in post_embeddings_df to match dst_node_id in sample_data
merged_data = pd.merge(
    sample_data,
    post_embeddings_df,
    left_on=['dst_node_id', 'timestamp'],
    right_on=['post_id', 'timestamp'],
    how='left'
)

# Check if any posts weren't found in the embeddings
missing_posts = merged_data[merged_data['embedding'].isna()]
if not missing_posts.empty:
    print(f"Warning: {len(missing_posts)} posts from sample data not found in embeddings")

# Display the merged data
merged_data

In [ ]:
post_embeddings_df[post_embeddings_df['post_id'] == 32]

In [ ]:
# Let's examine the eval_test_data object to see its properties
print("Number of interactions:", eval_test_data.num_interactions)
print("Number of unique nodes:", eval_test_data.num_unique_nodes)
print("Shape of src_node_ids:", eval_test_data.src_node_ids.shape)
print("Shape of dst_node_ids:", eval_test_data.dst_node_ids.shape)
print("Shape of node_interact_times:", eval_test_data.node_interact_times.shape)
print("Shape of edge_ids:", eval_test_data.edge_ids.shape)
print("Shape of labels:", eval_test_data.labels.shape)
print("Shape of idx:", eval_test_data.idx.shape)
print("Source max ID:", eval_test_data.src_max_id)

# Display a sample of the data
print("\nSample of the first 5 interactions:")
for i in range(min(5, eval_test_data.num_interactions)):
    print(f"Interaction {i}: src={eval_test_data.src_node_ids[i]}, dst={eval_test_data.dst_node_ids[i]}, time={eval_test_data.node_interact_times[i]}")

In [ ]:
# Filter posts that occur after June 1, 2023
cutoff_date = pd.to_datetime('2023-06-01')
filtered_post_embeddings = post_embeddings_df[pd.to_datetime(post_embeddings_df['timestamp'], unit='ms') > cutoff_date]

# Count how many posts have only 1 interaction after June 1
post_interaction_counts = filtered_post_embeddings['post_id'].value_counts()
single_interaction_posts = post_interaction_counts[post_interaction_counts == 1]
print(f"Number of posts after June 1 with only 1 interaction: {len(single_interaction_posts)}")
print(f"Percentage of posts after June 1 with only 1 interaction: {len(single_interaction_posts) / len(post_interaction_counts) * 100:.2f}%")

# Display the filtered dataframe
filtered_post_embeddings

In [ ]:
# Cell 10: Compare with Original CandidateEdgeSampler
# Create the original heuristic-based sampler for comparison
original_sampler = CandidateEdgeSampler(
    src_node_ids=full_data.src_node_ids, 
    dst_node_ids=full_data.dst_node_ids, 
    interact_times=full_data.node_interact_times
)

# Run evaluation with original sampler
print("Running evaluation with original candidate sampler...")
max_compare_samples = 50  # Small number for quick comparison

from evaluate_models_utils import evaluate_real as evaluate_with_original
# Note: You might need to modify this to limit the number of samples or handle different return values

# For comparison only - import the original evaluation function 
original_mrr = evaluate_with_original(
    model_name=args.model_name,
    model=model,
    neighbor_sampler=full_neighbor_sampler,
    evaluate_idx_data_loader=test_idx_data_loader, 
    evaluate_neg_edge_sampler=original_sampler,
    evaluate_data=eval_test_data,
    num_neighbors=args.num_neighbors,
    time_gap=args.time_gap
)

print(f"\nComparison of MRR scores:")
print(f"Embedding-based candidate generation: {test_mrr:.4f}")
print(f"Original heuristic-based generation: {original_mrr:.4f}")